<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 10. GPT-1 Generative Pre-Training — Improving Language Understanding by Generative Pre-Training

- **원 논문:** [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
- **저자 / 발표:** Alec Radford, Karthik Narasimhan, Tim Salimans, Ilya Sutskever · OpenAI Technical Report (2018)
- **난이도 / 예상 시간:** 중상급 · 약 85분
- **선수 지식:** Transformer decoder, causal LM loss, fine-tuning, 분류 cross entropy

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

작은 decoder-only Transformer를 causal language modeling으로 사전학습하고, 마지막 token 표현의 분류 head와 보조 LM 손실을 함께 미세조정한다. 미래 token 불변성과 사전학습/미세조정 metric을 통해 GPT-1의 두 단계 절차를 재현한다.

**원 논문과 다른 것**

GPT-1은 BooksCorpus(7천 권 이상)의 연속 텍스트로 12층, 768-hidden, 12-head, 약 1.17억 파라미터 Transformer를 사전학습한 뒤 여러 NLP benchmark에 미세조정했다. 여기서는 8개 고정 수열과 1층 24-hidden 모형을 사용한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.1, 식 (1): corpus token의 autoregressive likelihood L1 | `lm_inputs`/`lm_targets` shift와 causal LM cross entropy | LM loss 감소율을 assert하고 곡선으로 표시 |
| §3.1, 식 (2): multi-layer Transformer decoder h_l | `CausalSelfAttention`, `GPTBlock`, token+position embedding | logit shape와 미래 token에 대한 prefix 불변성 검사 |
| §3.1, 식 (2): P(u)=softmax(h_n W_e^T)의 embedding weight 재사용 | `TinyGPT.lm_logits`의 `F.linear(hidden, token_embedding.weight)` | LM vocabulary 축과 embedding parameter identity를 코드로 확인 |
| §3.2, 식 (3): 마지막 Transformer activation을 선형 분류기로 전달 | `classification_head(hidden[:, -1])` | 합성 분류 accuracy로 평가 |
| §3.2, 식 (4): L3 = L2 + λ·L1 보조 언어모델 목적 | `classification_loss + 0.2 * language_model_loss` | 두 loss가 유한한지와 최종 accuracy를 assert |
| §3.3 및 Figure 1: 구조화 입력을 ordered token sequence로 변환 | `sequences`가 BOS/content/EOS를 하나의 연속 token sequence로 구성 | 고정 길이와 경계 token을 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\mathcal{L}_{\mathrm{LM}}=-\sum_{t=1}^{T}
\log p(x_t\mid x_{<t}),\qquad
\mathcal{L}=\mathcal{L}_{\mathrm{task}}+\lambda\mathcal{L}_{\mathrm{LM}}
$$

- $\mathcal{L}_{\mathrm{LM}}$은 비라벨 문장의 다음 token 예측, $\mathcal{L}_{\mathrm{task}}$는 감독 목표입니다.
- 동일 Transformer decoder를 pretraining 후 작은 태스크에 fine-tuning하는 연결을 재현합니다.
- causal model, LM batch, task head, 두 loss를 따로 작성해 objective 경계를 보입니다.
- token logit은 $[B,T,V]$, task logit은 sequence representation으로부터 $[B,C]$가 됩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.1, 식 (1): corpus token의 autoregressive likelihood L1
- **노트북 구현:** `lm_inputs`/`lm_targets` shift와 causal LM cross entropy
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** LM loss 감소율을 assert하고 곡선으로 표시를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** GPT-1은 BooksCorpus(7천 권 이상)의 연속 텍스트로 12층, 768-hidden, 12-head, 약 1.17억 파라미터 Transformer를 사전학습한 뒤 여러 NLP benchmark에 미세조정했다. 여기서는 8개 고정 수열과 1층 24-hidden 모형을 사용한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

GPT-1의 절차는 (1) 레이블 없는 연속 텍스트에서 왼쪽 문맥만 보는 language model
likelihood $L_1$을 최대화하고, (2) 같은 Transformer를 task input에 맞게 미세조정하는
것이다. 지도 목적 $L_2$에 보조 LM 목적을 더한 $L_3=L_2+\lambda L_1$도 사용한다.
코드는 likelihood 최대화 대신 부호를 바꾼 cross-entropy를 최소화한다.

`nn.TransformerEncoder` 한 줄로 숨기지 않고, 식 (2)의 causal self-attention,
residual, LayerNorm, GELU feed-forward를 `CausalSelfAttention`과 `GPTBlock`으로
분리한다. GPT-1은 원 Transformer를 바탕으로 하므로 여기서는 residual 뒤에
LayerNorm을 두는 post-LN 순서를 쓴다. 언어모델 출력은 식 (2)처럼 token
embedding weight를 전치해 재사용한다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 1010
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())
BOS, EOS, VOCAB_SIZE = 1, 2, 16

### 구현 단계 2 · 논문 주장 확인 실험

- **원문 위치:** §3.1, 식 (1): corpus token의 autoregressive likelihood L1 / §3.3 및 Figure 1: 구조화 입력을 ordered token sequence로 변환
- **이 셀의 책임:** `lm_inputs`/`lm_targets` shift와 causal LM cross entropy / `sequences`가 BOS/content/EOS를 하나의 연속 token sequence로 구성
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** LM loss 감소율을 assert하고 곡선으로 표시 / 고정 길이와 경계 token을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,torch.cat,ACCELERATOR.move,torch.equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
content = torch.tensor(
    [
        [3, 4, 5, 6],
        [3, 4, 5, 7],
        [3, 5, 4, 6],
        [3, 5, 4, 7],
        [8, 9, 10, 11],
        [8, 9, 10, 12],
        [8, 10, 9, 11],
        [8, 10, 9, 12],
    ]
)
labels = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
bos = torch.full((len(content), 1), BOS, dtype=torch.long)
eos = torch.full((len(content), 1), EOS, dtype=torch.long)
sequences = torch.cat((bos, content, eos), dim=1)
lm_inputs, lm_targets = sequences[:, :-1], sequences[:, 1:]
content, labels, sequences, lm_inputs, lm_targets = ACCELERATOR.move(
    content, labels, sequences, lm_inputs, lm_targets
)

assert sequences.shape == (8, 6)
assert torch.all(sequences[:, 0] == BOS) and torch.all(sequences[:, -1] == EOS)
assert torch.equal(lm_inputs[:, 1:], lm_targets[:, :-1])

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.1, 식 (2): multi-layer Transformer decoder h_l / §3.1, 식 (2): P(u)=softmax(h_n W_e^T)의 embedding weight 재사용
- **이 셀의 책임:** `CausalSelfAttention`, `GPTBlock`, token+position embedding / `TinyGPT.lm_logits`의 `F.linear(hidden, token_embedding.weight)`
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** logit shape와 미래 token에 대한 prefix 불변성 검사 / LM vocabulary 축과 embedding parameter identity를 코드로 확인. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,Tensor.view,Tensor.transpose,torch.sqrt,torch.ones,torch.bool,Tensor.masked_fill,torch.softmax,nn.LayerNorm,F.gelu,nn.Embedding,nn.ModuleList,torch.arange,F.linear -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 14개 · 수식 9개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>tensor.view(*shape)</code></summary>

#### `tensor.view(*shape)`

- **역할:** 같은 저장공간을 보는 새로운 shape의 텐서를 만듭니다.
- **입력·반환:** 새 shape을 받고 view 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 연속 메모리의 축을 복사 없이 재해석합니다.
- **주의점:** stride가 호환되지 않으면 실패하므로 `reshape`나 `contiguous`가 필요합니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.view](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html)

</details>

<details>
<summary><code>tensor.transpose(dim0, dim1)</code></summary>

#### `tensor.transpose(dim0, dim1)`

- **역할:** 두 축의 위치를 맞바꿉니다.
- **입력·반환:** 두 축 번호를 받고 축이 교환된 view를 반환합니다.
- **이 셀에서 쓰는 이유:** attention 행렬곱 등에서 축 순서를 맞춥니다.
- **주의점:** 반환 텐서는 비연속일 수 있어 이후 `view` 전에 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
Y_{i_{\pi(1)},\ldots,i_{\pi(K)}}=X_{i_1,\ldots,i_K}
$$

- **기호:** $\pi$는 축의 새 순서를 나타내는 permutation입니다.
- **수식 → 코드:** `transpose`는 두 축만 바꾸고 `permute`는 모든 축의 새 순서를 명시합니다.
- **직관:** 원소 값을 바꾸지 않고 각 index가 의미하는 축의 위치만 다시 배치합니다.
- **shape 확인:** 예: `[B,T,H,D] -> [B,H,T,D]`; 결과는 비연속 view일 수 있습니다.
- **공식 문서:** [Tensor.transpose](https://docs.pytorch.org/docs/stable/generated/torch.transpose.html)

</details>

<details>
<summary><code>torch.sqrt(input)</code></summary>

#### `torch.sqrt(input)`

- **역할:** 각 원소의 제곱근을 계산합니다.
- **입력·반환:** 0 이상 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표준편차, 거리, diffusion schedule의 scale을 계산합니다.
- **주의점:** 음수 입력은 NaN을 만들며 0 근처 gradient가 매우 커질 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=\sqrt{x_i},\qquad x_i\ge 0
$$

- **기호:** $x_i$는 음수가 아닌 입력입니다.
- **수식 → 코드:** API가 제곱 또는 분산 scale을 원래 단위의 길이로 되돌립니다.
- **직관:** 제곱된 크기를 사람이 해석하기 쉬운 원래 scale로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sqrt](https://docs.pytorch.org/docs/stable/generated/torch.sqrt.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>tensor.masked_fill(mask, value)</code></summary>

#### `tensor.masked_fill(mask, value)`

- **역할:** mask가 참인 위치를 지정 값으로 바꿉니다.
- **입력·반환:** 불리언 mask와 값을 받고 새 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** padding이나 미래 token의 attention score를 선택에서 제외합니다.
- **주의점:** mask는 입력에 broadcast 가능해야 하고 softmax 전 값은 충분히 작아야 합니다.

##### 관련 수식과 코드 연결

$$
s'_{ij}=\begin{cases}s_{ij},&M_{ij}=0\\-\infty,&M_{ij}=1\end{cases}\quad\Rightarrow\quad\operatorname{softmax}(s')_{ij}=0\;\text{if }M_{ij}=1
$$

- **기호:** $s$는 score, $M$은 제외할 위치가 참인 mask입니다.
- **수식 → 코드:** attention에서는 mask 위치를 매우 작은 값으로 채운 뒤 softmax를 적용합니다.
- **직관:** 확률을 나중에 지우는 대신 정규화 분모에도 들어가지 못하게 합니다.
- **shape 확인:** mask는 score에 broadcast 가능해야 하며 보통 `[B, H, T_q, T_k]`를 사용합니다.
- **공식 문서:** [Tensor.masked_fill](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html)

</details>

<details>
<summary><code>torch.softmax(input, dim, dtype=None)</code></summary>

#### `torch.softmax(input, dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** attention score나 class logit을 정규화합니다.
- **주의점:** `dim`을 잘못 고르면 엉뚱한 축이 정규화되고 큰 logit은 포화될 수 있습니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [torch.softmax](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)

</details>

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

<details>
<summary><code>F.gelu(input, approximate=&#x27;none&#x27;)</code></summary>

#### `F.gelu(input, approximate='none')`

- **역할:** 입력을 확률적으로 gate하는 형태의 부드러운 GELU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** Transformer FFN의 비선형 변환을 구현합니다.
- **주의점:** 근사 방식에 따라 작은 수치 차이가 생길 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.gelu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.gelu.html)

</details>

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>nn.ModuleList(modules=None)</code></summary>

#### `nn.ModuleList(modules=None)`

- **역할:** 하위 모듈 목록을 등록 가능한 형태로 보관합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.ModuleList](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>F.linear(input, weight, bias=None)</code></summary>

#### `F.linear(input, weight, bias=None)`

- **역할:** 명시적으로 전달한 weight와 bias로 affine 변환을 계산합니다.
- **입력·반환:** 입력·weight·선택적 bias를 받고 마지막 feature 축이 바뀐 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 별도 `nn.Linear` 객체 없이 공유·재사용 weight의 선형 변환을 드러냅니다.
- **주의점:** weight shape은 `[out_features, in_features]`이고 transpose 방향에 주의합니다.
- **관련 수식:** 이 셀의 `nn.Linear(in_features, out_features, bias=True)`와 같은 수식 계약을 사용합니다. 중복 표시는 생략했습니다.
- **공식 문서:** [F.linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.linear.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_key_value = nn.Linear(
            d_model,
            3 * d_model,
            bias=False,
        )
        self.output_projection = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

    def _split_heads(self, tensor):
        batch, length, width = tensor.shape
        split = tensor.view(
            batch,
            length,
            self.num_heads,
            width // self.num_heads,
        )
        return split.transpose(1, 2)

    def _merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        merged = tensor.transpose(1, 2).contiguous()
        return merged.view(
            batch,
            length,
            self.num_heads * self.head_dim,
        )

    def forward(self, hidden):
        query, key, value = self.query_key_value(hidden).chunk(3, dim=-1)
        query = self._split_heads(query)
        key = self._split_heads(key)
        value = self._split_heads(value)
        scores = query @ key.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)
        length = hidden.shape[1]
        blocked = torch.ones(
            length,
            length,
            dtype=torch.bool,
            device=hidden.device,
        ).triu(diagonal=1)
        scores = scores.masked_fill(
            blocked.view(1, 1, length, length),
            torch.finfo(scores.dtype).min,
        )
        weights = torch.softmax(scores, dim=-1)
        context = self._merge_heads(weights @ value)
        return self.output_projection(context), weights


class GPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention = CausalSelfAttention(d_model, num_heads)
        self.attention_norm = nn.LayerNorm(d_model)
        self.feed_forward_input = nn.Linear(d_model, d_ff)
        self.feed_forward_output = nn.Linear(d_ff, d_model)
        self.feed_forward_norm = nn.LayerNorm(d_model)

    def forward(self, hidden):
        attention_output, weights = self.attention(hidden)
        hidden = self.attention_norm(hidden + attention_output)
        expanded = self.feed_forward_input(hidden)
        feed_forward_output = self.feed_forward_output(F.gelu(expanded))
        hidden = self.feed_forward_norm(hidden + feed_forward_output)
        return hidden, weights


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, max_length, d_model=24, num_heads=4):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.blocks = nn.ModuleList([GPTBlock(d_model, num_heads, d_ff=2 * d_model)])
        self.classification_head = nn.Linear(d_model, 2)

    def encode(self, token_ids):
        length = token_ids.shape[1]
        positions = torch.arange(
            length,
            device=token_ids.device,
        ).unsqueeze(0)
        hidden = self.token_embedding(token_ids)
        hidden = hidden + self.position_embedding(positions)
        for block in self.blocks:
            hidden, _ = block(hidden)
        return hidden

    def lm_logits(self, token_ids):
        hidden = self.encode(token_ids)
        return F.linear(hidden, self.token_embedding.weight)

    def classification_logits(self, token_ids):
        return self.classification_head(self.encode(token_ids)[:, -1])


torch.manual_seed(SEED)
model = TinyGPT(VOCAB_SIZE, max_length=sequences.shape[1]).to(DEVICE)
assert model.lm_logits(lm_inputs).shape == (*lm_inputs.shape, VOCAB_SIZE)
assert model.classification_logits(sequences).shape == (len(sequences), 2)
assert len(model.blocks) == 1

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3.1, 식 (1): corpus token의 autoregressive likelihood L1
- **이 셀의 책임:** `lm_inputs`/`lm_targets` shift와 causal LM cross entropy
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** LM loss 감소율을 assert하고 곡선으로 표시. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam,Module.parameters,Module.train,Optimizer.zero_grad,F.cross_entropy,Tensor.reshape,Tensor.backward,Optimizer.step,Tensor.detach,Module.eval,torch.inference_mode -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 11개 · 수식 5개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>tensor.reshape(*shape)</code></summary>

#### `tensor.reshape(*shape)`

- **역할:** 원소 수를 유지하며 텐서 shape을 바꿉니다.
- **입력·반환:** 새 shape을 받고 가능하면 view, 아니면 복사된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch·head·sequence 축을 논문 수식의 shape에 맞춥니다.
- **주의점:** 원소 수가 같아야 하며 메모리 공유 여부에 의존하면 안 됩니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.reshape](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
pretrain_optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
pretrain_losses = []
for _ in range(70):
    model.train()
    pretrain_optimizer.zero_grad()
    logits = model.lm_logits(lm_inputs)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), lm_targets.reshape(-1))
    loss.backward()
    pretrain_optimizer.step()
    pretrain_losses.append(float(loss.detach()))

assert pretrain_losses[-1] < 0.45 * pretrain_losses[0]

model.eval()
original = lm_inputs[:1].clone()
changed = original.clone()
changed[:, 3:] = changed.new_tensor([[13, 14]])
with torch.inference_mode():
    original_logits = model.lm_logits(original)
    changed_logits = model.lm_logits(changed)
prefix_change = (original_logits[:, :3] - changed_logits[:, :3]).abs().max().item()
assert prefix_change < 1e-6
print(
    f"pretrain_loss={pretrain_losses[0]:.3f}->{pretrain_losses[-1]:.3f}, "
    f"prefix_change={prefix_change:.2e}"
)

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** §3.2, 식 (3): 마지막 Transformer activation을 선형 분류기로 전달 / §3.2, 식 (4): L3 = L2 + λ·L1 보조 언어모델 목적
- **이 셀의 책임:** `classification_head(hidden[:, -1])` / `classification_loss + 0.2 * language_model_loss`
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 합성 분류 accuracy로 평가 / 두 loss가 유한한지와 최종 accuracy를 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
finetune_optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
supervised_losses, auxiliary_lm_losses = [], []
for _ in range(60):
    model.train()
    finetune_optimizer.zero_grad()
    class_logits = model.classification_logits(sequences)
    lm_logits = model.lm_logits(lm_inputs)
    supervised_loss = F.cross_entropy(class_logits, labels)
    auxiliary_lm_loss = F.cross_entropy(
        lm_logits.reshape(-1, VOCAB_SIZE), lm_targets.reshape(-1)
    )
    joint_loss = supervised_loss + 0.2 * auxiliary_lm_loss
    joint_loss.backward()
    finetune_optimizer.step()
    supervised_losses.append(float(supervised_loss.detach()))
    auxiliary_lm_losses.append(float(auxiliary_lm_loss.detach()))

model.eval()
with torch.inference_mode():
    predictions = model.classification_logits(sequences).argmax(dim=-1)
classification_accuracy = (predictions == labels).float().mean().item()
assert classification_accuracy >= 0.875
assert math.isfinite(supervised_losses[-1])
assert math.isfinite(auxiliary_lm_losses[-1])
print(f"fine-tune accuracy={classification_accuracy:.1%}")

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1, 식 (1): corpus token의 autoregressive likelihood L1 / §3.2, 식 (3): 마지막 Transformer activation을 선형 분류기로 전달 / §3.2, 식 (4): L3 = L2 + λ·L1 보조 언어모델 목적
- **이 셀의 책임:** `lm_inputs`/`lm_targets` shift와 causal LM cross entropy / `classification_head(hidden[:, -1])` / `classification_loss + 0.2 * language_model_loss`
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** LM loss 감소율을 assert하고 곡선으로 표시 / 합성 분류 accuracy로 평가 / 두 loss가 유한한지와 최종 accuracy를 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
axes[0].plot(pretrain_losses, label="pretrain LM")
axes[0].plot(auxiliary_lm_losses, label="fine-tune auxiliary LM")
axes[0].set(title="generative objectives", xlabel="step", ylabel="cross entropy")
axes[0].legend()
axes[1].bar(
    ["error", "accuracy"],
    [1 - classification_accuracy, classification_accuracy],
)
axes[1].set(title="downstream classification", ylim=(0, 1.05))
plt.tight_layout()
plt.show()

## 해석 체크

보조 LM loss가 미세조정 중 어떤 regularizer 역할을 하는지, 그리고 이 실습의
`classification_logits`가 GPT-1 Figure 1의 task-aware input transform 전체를
재현하지 않는 이유를 적어 보세요.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `GPT-1은 BooksCorpus(7천 권 이상)의 연속 텍스트로 12층, 768-hidden, 12-head, 약 1.17억 파라미터 Transformer를 사전학습한 뒤 여러 NLP benchmark에 미세조정했다. 여기서는 8개 고정 수열과 1층 24-hidden 모형을 사용한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.